In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/FMCG Project/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Reading Raw data from Source**

In [0]:
products_base_path = f"/Volumes/fmcg/default/child_company/fullload/full_load/products/products.csv"
dbutils.fs.ls(products_base_path)

In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
        .option("inferSchema", True) \
            .load(products_base_path)

In [0]:
display(df)

### **Addition of metadata info**

In [0]:
df = df.withColumn("dataload_ts", F.current_timestamp()) \
    .select("*", "_metadata.file_name", "_metadata.file_size")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.write \
    .format("delta") \
        .option("delta.enableChangeDataFeed", True) \
            .mode("overwrite") \
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")